# FahMai Telephone Directory v5
## 5-Stage Hybrid Architecture (Rule-based + LLM)


# 1. Setup


In [1]:
!pip install -q openai pandas tqdm

import os, json, re, time, warnings
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI

warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    TYPHOON_API_KEY = UserSecretsClient().get_secret("TYPHOON_API_KEY")
except Exception:
    try:
        from google.colab import userdata
        TYPHOON_API_KEY = userdata.get('typhoon')
    except Exception:
        TYPHOON_API_KEY = os.environ.get('TYPHOON_API_KEY', '')

MODEL      = 'typhoon-v2.5-30b-a3b-instruct'
MAX_TOKENS = 2048

client = OpenAI(api_key=TYPHOON_API_KEY, base_url='https://api.opentyphoon.ai/v1')

# 2. Data Loading


In [2]:
DATA_DIR   = Path('/kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/')

df_emp = pd.read_csv(DATA_DIR / 'employees.csv').fillna('')
df_questions = pd.read_csv(DATA_DIR / 'questions.csv')

df_emp['Phone Extension'] = df_emp['Phone Extension'].apply(
    lambda x: str(int(float(x))) if x not in ('',None) and str(x).replace('.','').isdigit() else str(x))

N_EMP = len(df_emp)
print(f"Employees: {N_EMP}, Questions: {len(df_questions)}")

Employees: 1995, Questions: 300


# 3. Pipeline: Stage 1 (Intent Router)


In [3]:
def stage1_intent_router(question, lang):
    q = question.lower()
    
    # 1. Field not in directory
    forbiddens = [
        r'\bเงินเดือน\b', r'\bsalary\b', r'\bอายุ\b', r'\bage\b', r'\bศาสนา\b', r'\breligion\b',
        r'\bโบนัส\b', r'\bbonus\b', r'\bประเมิน\b', r'\bperformance\b', r'\bเลื่อนตำแหน่ง\b',
        r'\bpromotion\b', r'\bลงโทษ\b', r'\bdisciplinary\b', r'\bวันเกิด\b', r'\bbirthday\b',
        r'\bกี่ปี\b', r'\bน้ำหนัก\b', r'\bส่วนสูง\b', r'\bweight\b', r'\bheight\b',
        r'\bแฟน\b', r'\bครอบครัว\b', r'\bfamily\b', r'\bmarried\b', r'\bแต่งงาน\b'
    ]
    if any(re.search(p, q) for p in forbiddens):
        return 'ไม่สามารถให้ข้อมูลนี้ได้' if lang == 'th' else 'cannot provide this information'
        
    # 2. Opinion / Speculation
    opinions = [
        r'\bเก่งที่สุด\b', r'\bbest\b', r'\bโปรโมท\b', r'\bควร\b', r'\bshould\b',
        r'\bแนะนำ\b', r'\brecommend\b', r'\bคิดว่า\b', r'\bopinion\b', r'\bเหมาะสม\b',
        r'\bใครดี\b', r'\bหล่อ\b', r'\bสวย\b', r'\bนิสัย\b'
    ]
    if any(re.search(p, q) for p in opinions):
        return 'ไม่สามารถให้ความเห็นได้' if lang == 'th' else 'cannot offer an opinion'
        
    # 3. External Company
    externals = [
        r'\bsamsung\b', r'\bapple\b', r'\blg\b', r'\bdtac\b', r'\bais\b', r'\btrue\b',
        r'\btoyota\b', r'\bhonda\b', r'\bgoogle\b', r'\bshopee\b', r'\blazada\b',
        r'\bซัมซุง\b', r'\bแอปเปิล\b', r'\bแอปเปิ้ล\b', r'\bกูเกิล\b', r'\bช้อปปี้\b', r'\bลาซาด้า\b'
    ]
    if any(re.search(p, q) for p in externals):
        if not re.search(r'\bfahmai\b|ฟ้าใหม่', q):
            return 'ไม่ใช่ข้อมูลของฟ้าใหม่' if lang == 'th' else 'not a FahMai record'
            
    # 4. Prompt injection
    injections = [
        r'\bignore previous\b', r'\bforget\b', r'\bsystem prompt\b',
        r'\bคำสั่ง\b', r'\bact as\b', r'\bpretend\b', r'\boverride\b', r'\bเปลี่ยนคำตอบ\b'
    ]
    if any(re.search(p, q) for p in injections):
        return 'ขอปฏิเสธคำขอ' if lang == 'th' else 'request declined'

    return None

# 4. Pipeline: Stage 2 (Deterministic Fast-Path)


In [4]:
LEVEL_SUFFIXES = {
    'VP': 'VP', 'EVP': 'VP', 'FL': 'Lead', 'DIR': 'Director',
    'MGR': 'Manager', 'LEAD': 'Lead', 'IC': 'IC'
}

def decode_position_code(code_str):
    code_str = code_str.strip().upper()
    for suffix, level in sorted(LEVEL_SUFFIXES.items(), key=lambda x: -len(x[0])):
        if code_str.endswith(suffix) and len(code_str) > len(suffix):
            return code_str[:-len(suffix)], level
    return None, None

def stage2_fast_path(question, lang, df):
    q_up = question.upper()
    q_low = question.lower()
    
    # 1. Exact Name Match (Highest confidence)
    for _, r in df.iterrows():
        fn_th = str(r['First Name Thai']).strip()
        ln_th = str(r['Last Name Thai']).strip()
        fn_en = str(r['First Name English']).strip()
        ln_en = str(r['Last Name English']).strip()
        
        if fn_th and ln_th and fn_th in question and ln_th in question:
            return df[(df['First Name Thai'] == fn_th) & (df['Last Name Thai'] == ln_th)]
        if fn_en and ln_en and fn_en.lower() in q_low and ln_en.lower() in q_low:
            return df[(df['First Name English'] == fn_en) & (df['Last Name English'] == ln_en)]

    # 2. GM
    if 'gm ' in q_low or 'general manager' in q_low or 'gm' in q_low:
        brand_map = {'สายฟ้า':'SAIFAH', 'ดาวเหนือ':'DAONUEA', 'วงโคจร':'WONGKHOJON', 'จุดเชื่อม':'JUDCHUEM', 'คลื่นเสียง':'KLUENSIANG'}
        for th, en in brand_map.items():
            if th in q_low or en.lower() in q_low:
                return df[df['Position in English'].str.contains(f'GENERAL MANAGER OF {en}', case=False, na=False)]

    # 3. Secretary
    if any(w in q_low for w in ['เลขา', 'secretary', 'ea of', 'assistant to']):
        boss_m = re.search(r'\b([A-Z]{2,}(?:VP|PM|DG|CX|FL|QA|BKK|UPC|ACC|CEO|CFO|CTO|CMO|COO|CHRO|CPO))\b', q_up)
        if boss_m:
            boss = boss_m.group(1)
            return df[df['Position in English'].str.contains(f'SECRETARY OF {boss}|EXECUTIVE ASSISTANT TO {boss}', case=False, na=False)]

    # 4. Unit / Section Code Exact Match
    exec_m = re.search(r'\b(TEC|OPS|HR|FIN|MKT|SF)-EXEC\b', q_up)
    if exec_m:
        return df[df['Section'] == exec_m.group(1)]

    code_m = re.search(r'\b([A-Z]{2,5}(?:VP|FL|DIR|MGR|PM|DG|CX|QA|BKK|UPC|ACC|FP))\b', q_up)
    if code_m:
        code = code_m.group(1)
        hits = df[df['Unit'] == code]
        if len(hits) > 0: return hits
        dept, lvl = decode_position_code(code)
        if dept and lvl:
            hits = df[(df['Department'] == dept) & (df['Position Level'] == lvl)]
            if len(hits) > 0: return hits

    # 5. Simple Count Bypass
    is_count = any(w in q_low for w in ['กี่คน', 'จำนวน', 'ทั้งหมดกี่', 'how many', 'count of', 'number of'])
    if is_count:
        if not any(w in q_low for w in ['ชื่อ', 'ตำแหน่ง', 'เลขา', 'vp', 'manager', 'lead', 'ic']):
            all_sections = [x for x in df['Section'].unique() if x]
            for sec in sorted(all_sections, key=len, reverse=True):
                if sec and sec in q_up:
                    c = len(df[df['Section']==sec])
                    return f"มีพนักงานในหน่วย {sec} ทั้งหมด {c} คน" if lang=='th' else f"There are {c} employees in {sec}."
            
            all_units = [x for x in df['Unit'].unique() if x]
            for u in sorted(all_units, key=len, reverse=True):
                if u and len(u)>2 and u in q_up:
                    c = len(df[df['Unit']==u])
                    return f"มีพนักงานในหน่วย {u} ทั้งหมด {c} คน" if lang=='th' else f"There are {c} employees in {u}."
            
            all_depts = [x for x in df['Department'].unique() if x]
            for d in sorted(all_depts, key=len, reverse=True):
                if d and d in q_up:
                    c = len(df[df['Department']==d])
                    return f"แผนก {d} มีทั้งหมด {c} คน" if lang=='th' else f"There are {c} employees in department {d}."

    # 6. Contact exact (Extension)
    ext_m = re.search(r'\b(\d{4})\b', q_low)
    if ext_m and any(w in q_low for w in ['เบอร์', 'โทร', 'ext', 'extension']):
        hits = df[df['Phone Extension'] == ext_m.group(1)]
        if len(hits) > 0: return hits

    # 7. Nickname + Dept
    depts = [d for d in df['Department'].unique() if d]
    found_dept = None
    for d in sorted(depts, key=len, reverse=True):
        if re.search(r'\b' + d + r'\b', q_up):
            found_dept = d
            break
    if found_dept:
        dept_emps = df[df['Department'] == found_dept]
        for _, r in dept_emps.iterrows():
            nk_th = str(r['Nickname Thai']).strip()
            nk_en = str(r['Nickname English']).strip()
            if nk_th and nk_th in question:
                return dept_emps[dept_emps['Nickname Thai'] == nk_th]
            if nk_en and re.search(r'\b' + re.escape(nk_en).lower() + r'\b', q_low):
                return dept_emps[dept_emps['Nickname English'].str.lower() == nk_en.lower()]

    # 8. Fruits mapping
    if 'ผลไม้' in q_low or 'fruit' in q_low:
        fruits = ['APPLE','BERRY','CHERRY','GRAPE','KIWI','LEMON','LIME','MANGO','MELON','OLIVE','PEACH','PLUM','ORANGE',
                  'แอปเปิ้ล','เบอร์รี่','เชอร์รี่','องุ่น','กีวี่','มะนาว','มะม่วง','แตงโม','พีช','ส้ม','มะกอก']
        mask = df['Nickname English'].str.upper().isin(fruits) | df['Nickname Thai'].isin(fruits)
        hits = df[mask]
        if len(hits) > 0: return hits

    # 9. Top-level Executives
    if 'สูงสุด' in q_low or 'ดูแล' in q_low or 'ผู้บริหาร' in q_low:
        if 'tech' in q_low or 'เทคโนโลยี' in q_low: return df[df['Unit'] == 'CTO']
        if 'operation' in q_low or 'ปฏิบัติการ' in q_low: return df[df['Unit'] == 'COO']
        if 'hr' in q_low or 'บุคลากร' in q_low: return df[df['Unit'] == 'CHRO']
        if 'fin' in q_low or 'เงิน' in q_low: return df[df['Unit'] == 'CFO']
        if 'mkt' in q_low or 'ตลาด' in q_low: return df[df['Unit'] == 'CMO']
        if 'prod' in q_low or 'ผลิต' in q_low: return df[df['Unit'] == 'CPO']

    return None

# 5. Pipeline: Stage 3 (Pandas Planner)


In [5]:
def stage3_pandas_planner(question, df):
    prompt = f'''You are an expert Python data analyst. The dataframe `df` has columns:
'First Name Thai', 'Last Name Thai', 'First Name English', 'Last Name English', 'Nickname Thai', 'Nickname English', 'Department', 'Unit', 'Section', 'Position in English', 'Phone Extension', 'Email Address', 'Office Location', 'Branch', 'Position Level'

Write ONLY a valid Python Pandas expression (one line) that extracts the answer from `df`.
RULES:
1. Output ONLY the code. No markdown formatting, no explanations.
2. The code must evaluate to a DataFrame, Series, or an Integer.
3. For counts, use len(df[...]).
4. Use case-insensitive contains for text search: df[df['Col'].str.contains('text', case=False, na=False)]

EXAMPLES:
Q: ใครคือ RETVP
df[df['Unit'] == 'RETVP']

Q: แผนก HR มีกี่คน
len(df[df['Department'] == 'HR'])

Q: {question}
'''
    messages = [{'role':'system', 'content':'/nothink'}, {'role':'user', 'content':prompt}]
    try:
        resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0, max_tokens=150)
        code = resp.choices[0].message.content.strip()
        code = re.sub(r'^```python\s*|```$', '', code, flags=re.MULTILINE).strip()
        
        result = eval(code)
        if isinstance(result, pd.DataFrame) and len(result) == 0:
            return None
        return result
    except Exception as e:
        print(f"Stage 3 eval error: {e}")
        return None

# 6. Pipeline: Stage 4 (Answer Formatter)


In [6]:
def stage4_formatter(question, hits, lang):
    # Handle scalar answers directly
    if isinstance(hits, (int, str)):
        prompt = f'''Answer the user's question concisely using this factual answer: {hits}
Rules:
- Answer in {'Thai' if lang=='th' else 'English'} language ONLY.
- Be direct and natural.

Question: {question}
Answer:'''
    else:
        # Handle DataFrames/Series
        if isinstance(hits, pd.DataFrame):
            records = hits.to_dict('records')
        elif isinstance(hits, pd.Series):
            records = [hits.to_dict()]
        else:
            records = []
            
        clean_records = [{k:v for k,v in r.items() if v} for r in records][:10]
        
        prompt = f'''Answer the user's question based ONLY on the provided JSON data.
Rules:
- Answer in {'Thai' if lang=='th' else 'English'} language ONLY.
- Keep the answer highly concise and direct.
- Do NOT use numbered lists.
- Do NOT dump all department members if not explicitly asked.
- If the question asks for a name, provide both First and Last name.

JSON Data:
{json.dumps(clean_records, ensure_ascii=False)}

Question: {question}
Answer:'''

    messages = [{'role':'system', 'content':'/nothink'}, {'role':'user', 'content':prompt}]
    try:
        resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0, max_tokens=256)
        ans = resp.choices[0].message.content.strip()
        ans = re.sub(r'<think>.*?</think>', '', ans, flags=re.DOTALL).strip()
        return ans
    except Exception:
        return 'ไม่พบข้อมูล' if lang == 'th' else 'no record found'

# 7. Pipeline: Stage 5 (Compliance Guard)


In [7]:
def stage5_compliance(question, answer, hits, lang):
    q_low = question.lower()
    ans_low = answer.lower()
    
    # 1. Blank Nickname Guard
    if 'ชื่อเล่น' in q_low or 'nickname' in q_low:
        has_nick = False
        if isinstance(hits, pd.DataFrame) and len(hits) > 0:
            for _, r in hits.iterrows():
                if str(r.get('Nickname Thai', '')).strip() or str(r.get('Nickname English', '')).strip():
                    has_nick = True
                    break
        elif isinstance(hits, pd.Series):
            if str(hits.get('Nickname Thai', '')).strip() or str(hits.get('Nickname English', '')).strip():
                has_nick = True
        
        if not has_nick:
            return 'ไม่มีชื่อเล่นในระบบ' if lang == 'th' else 'nickname not listed'
            
    # 2. Extract refusal phrases accurately
    if 'ไม่พบข้อมูล' in ans_low or 'no record found' in ans_low:
        return 'ไม่พบข้อมูล' if lang == 'th' else 'no record found'
    if 'ไม่สามารถ' in ans_low or 'cannot provide' in ans_low:
        return 'ไม่สามารถให้ข้อมูลนี้ได้' if lang == 'th' else 'cannot provide this information'
    
    return answer

# 8. Orchestration & Inference


In [8]:
def process_question(q_id, question, lang, verbose=False):
    fallback = 'ไม่พบข้อมูล' if lang == 'th' else 'no record found'
    
    # Stage 1
    intent = stage1_intent_router(question, lang)
    if intent:
        if verbose: print(f"[{q_id}] Stage 1 (Intent) -> {intent}")
        return intent
        
    hits = None
    
    # Stage 2
    fast_hits = stage2_fast_path(question, lang, df_emp)
    if fast_hits is not None:
        if isinstance(fast_hits, str): 
            if verbose: print(f"[{q_id}] Stage 2 (Direct String) -> {fast_hits}")
            return fast_hits
        if isinstance(fast_hits, pd.DataFrame) and len(fast_hits) == 0:
            if verbose: print(f"[{q_id}] Stage 2 (Empty DF) -> {fallback}")
            return fallback
        if verbose: print(f"[{q_id}] Stage 2 (Hit DF length {len(fast_hits)})")
        hits = fast_hits
        
    # Stage 3
    if hits is None:
        hits = stage3_pandas_planner(question, df_emp)
        if hits is None or (isinstance(hits, pd.DataFrame) and len(hits) == 0):
            if verbose: print(f"[{q_id}] Stage 3 (Empty/Failed) -> {fallback}")
            return fallback
        if verbose: print(f"[{q_id}] Stage 3 (Pandas Eval Hit)")
            
    # Stage 4
    ans = stage4_formatter(question, hits, lang)
    
    # Stage 5
    final_ans = stage5_compliance(question, ans, hits, lang)
    if verbose: print(f"[{q_id}] Final -> {final_ans}")
    
    return final_ans

results = []
print(f"Inference: {len(df_questions)} questions | Model: {MODEL}")

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc='v5'):
    q_id     = row['id']
    question = row['question']
    language = row['language'] if 'language' in row.index else 'th'
    
    answer = process_question(q_id, question, language, verbose=False)
    results.append({'id': q_id, 'response': answer})

submission_df = pd.DataFrame(results)
submission_df.to_csv('submission_FTD.csv', index=False, encoding='utf-8')
print(f"Saved {len(submission_df)} rows -> submission_FTD.csv")
display(submission_df.head(10))

Inference: 300 questions | Model: typhoon-v2.5-30b-a3b-instruct


v5:   0%|          | 0/300 [00:00<?, ?it/s]

Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: 'Education'
Stage 3 eval error: 'Education Level'
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: single positional indexer is out-of-bounds
Stage 3 eval error: 'Reporting To'
Stage 3 eval error: 'Reporting Chain'
Stage 3 eval error: single positional 

,id,response
0,g001,Wiriya Chanchai
1,g002,คึกฤทธิ์ บุษราคัมวงศ์
2,g004,ไพโรจน์ มหากุล
3,g005,ไม่พบข้อมูล
4,g007,มาลี อมรทอง
5,g008,เรืองศักดิ์ เทพเกียรติกำจร
6,g009,ดาริกา อาวุทธ์ดี
7,g011,no record found
8,g012,คะวัง กอบสุขรัตน์
9,g014,ณัฐกานต์ ศรีอารมณ์ดี


# 9. Local Grade


In [9]:
import subprocess
grade_script = DATA_DIR / 'grade.py'
train_labels = DATA_DIR / 'train_labels.json'

if grade_script.exists() and train_labels.exists():
    res = subprocess.run(
        ['python', str(grade_script), 'submission_FTD.csv', str(train_labels)],
        capture_output=True, text=True, encoding='utf-8')
    print("="*60)
    print(res.stdout)
    if res.stderr:
        print("STDERR:", res.stderr[:300])
else:
    print("grade.py not found in", DATA_DIR)

Scored 158 items against /kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/train_labels.json
Passed: 61/158 = 38.6%

Bucket                             pass/total    rate
--------------------------------------------------------
nickname_grid                    4/      17   23.5%
refuse                           4/      15   26.7%
evp_secretary                    7/       9   77.8%
vp_identity                      0/       9    0.0%
casual_name_lookup               4/       9   44.4%
evp_identity_by_code             7/       8   87.5%
evp_identity_by_description      2/       8   25.0%
name_lookup                      8/       8  100.0%
dept_listing_medium              0/       8    0.0%
dept_member_count                5/       7   71.4%
dept_listing_small               0/       6    0.0%
extension_reverse                4/       5   80.0%
hard_nickname_variant            5/       5  100.0%
evp_vs_vp_disambig               2/       4   50.0%
email_mobile_